# 01 — Comprehensive Data Audit & Exploratory Data Analysis (EDA)
**Project:** Insurance Claim Fraud Detection — Major Project Phase 1  
**Objective:** Perform a rigorous statistical audit of the claims dataset, detect data quality defects (e.g. `'?'` sentinel values, zero variance fields), inspect the 24.7% class imbalance, examine correlation structures, and validate claim consistency.

In [ ]:
from pathlib import Path

import sys

import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns



# Add src to path

sys.path.insert(0, str(Path.cwd().parent))

from src.config import RAW_DATA_PATH, TARGET_COL, TARGET_MAPPING

from src.data_loader import load_raw_data, parse_dates_and_target



sns.set_theme(style='whitegrid')

pd.set_option('display.max_columns', 50)

## 1. Load Data with Explicit '?' Sentinel Value Handling

In [ ]:
# Load claims raw data

raw_df = load_raw_data(RAW_DATA_PATH)

print(f'Total Claim Records: {len(raw_df):,}')

print(f'Total Attributes: {raw_df.shape[1]}')

raw_df.head(3)

## 2. Missing Value & Data Quality Audit
Notice how `collision_type`, `property_damage`, and `police_report_available` contained `'?'` characters in the original data. We now recognize them as real missing values.

In [ ]:
missing_summary = pd.DataFrame({

    'missing_count': raw_df.isna().sum(),

    'missing_pct': (raw_df.isna().mean() * 100).round(2)

}).query('missing_count > 0').sort_values('missing_count', ascending=False)



print('Features with Missing Values:')

display(missing_summary)

## 3. Fraud Target Distribution
Fraud claims represent ~24.7% of the dataset. Accuracy alone will be misleading; we must track Precision, Recall, F1, and PR-AUC.

In [ ]:
clean_df = parse_dates_and_target(raw_df)

target_counts = clean_df[TARGET_COL].value_counts()



fig, axes = plt.subplots(1, 2, figsize=(12, 4))

target_counts.plot(kind='bar', color=['#2b5c8f', '#d9534f'], ax=axes[0])

axes[0].set_title('Target Class Counts (0: Non-Fraud, 1: Fraud)')

axes[0].set_xlabel('Fraud Reported')

axes[0].set_ylabel('Number of Claims')



axes[1].pie(target_counts, labels=['Non-Fraud (75.3%)', 'Fraud (24.7%)'], autopct='%1.1f%%', colors=['#2b5c8f', '#d9534f'], startangle=140)

axes[1].set_title('Class Imbalance Proportion')

plt.tight_layout()

plt.show()

## 4. Financial Component Consistency Check
Validating that `total_claim_amount == injury_claim + property_claim + vehicle_claim`.

In [ ]:
component_sum = clean_df['injury_claim'] + clean_df['property_claim'] + clean_df['vehicle_claim']

discrepancy = (clean_df['total_claim_amount'] - component_sum).abs()

print(f'Max discrepancy between total claim and component sum: ${discrepancy.max():.2f}')

assert discrepancy.max() == 0, 'Discrepancy detected in claim accounting!'

## 5. Fraud Rate Across Key Categorical Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=clean_df, x='incident_severity', hue=TARGET_COL, palette='Set1', ax=axes[0, 0])

axes[0, 0].set_title('Claims by Incident Severity')



sns.countplot(data=clean_df, x='incident_type', hue=TARGET_COL, palette='Set1', ax=axes[0, 1])

axes[0, 1].set_title('Claims by Incident Type')

axes[0, 1].tick_params(axis='x', rotation=30)



sns.countplot(data=clean_df, x='authorities_contacted', hue=TARGET_COL, palette='Set1', ax=axes[1, 0])

axes[1, 0].set_title('Claims by Authorities Contacted')



sns.countplot(data=clean_df, x='insured_sex', hue=TARGET_COL, palette='Set1', ax=axes[1, 1])

axes[1, 1].set_title('Claims by Insured Sex')

plt.tight_layout()

plt.show()